In [70]:
from google.cloud import storage
import os
from dotenv import load_dotenv
from io import BytesIO
import pandas as pd

load_dotenv()

True

In [71]:
GCS_BUCKET = os.getenv('GCS_BUCKET').strip()
GCS_MEDAL_FETCH = os.getenv('GCS_MEDAL_FETCH').strip()
SEASON = int(os.getenv('SEASON'))
BLOB_FETCH = f"{GCS_MEDAL_FETCH}/season={SEASON}/all_session_laps.parquet"

In [72]:
def load_parquet_from_gcs(bucket, blob_path):
    if not bucket:
        raise ValueError("[ERROR] Bucket Name is Required")
    if not blob_path:
        raise ValueError("[ERROR] Blob Path is Required")

    client = storage.Client()
    bucket = client.bucket(bucket)
    blob = bucket.blob(blob_path)

    if not blob.exists():
        raise FileNotFoundError(f"[ERROR] Blob {blob_path} does not exist")
    
    print(f"[INFO] Blob exists: {blob.name}")
    df = pd.read_parquet(BytesIO(blob.download_as_bytes()))
    return df


In [73]:
df = load_parquet_from_gcs(GCS_BUCKET, BLOB_FETCH)

[INFO] Blob exists: silver/season=2026/all_session_laps.parquet


In [74]:
df = df.loc[df["session_name"] == "Qualifying"]

In [75]:
df["date"] = df["date_start"].dt.strftime("%Y-%m-%d")
df["time"] = df["date_start"].dt.strftime("%H:%M:%S")

In [76]:
columns_to_drop = [
    "lap_number",
    "segments_sector_1",
    "segments_sector_2",
    "segments_sector_3",
    "date_start"
]

df = df.drop(columns=columns_to_drop)

In [77]:
df = df.sort_values(
    by=["session_key", "driver_number", "lap_duration"],
    ascending=[True, True, True]
)
df = df.drop_duplicates(subset=["session_key", "driver_number"], keep="first")
df = df.sort_values(
    by=["session_key", "lap_duration"],
    ascending=[True, True]
)

In [78]:
td = pd.to_timedelta(df["lap_duration"], unit="s")

minutes = (td.dt.total_seconds() // 60).astype(int)
seconds = (td.dt.total_seconds() % 60).astype(int)
milliseconds = (td.dt.microseconds // 1000).astype(int)

df["lap_duration_fmt"] = (
    minutes.astype(str).str.zfill(2) + ":" +
    seconds.astype(str).str.zfill(2) + "." +
    milliseconds.astype(str).str.zfill(3)
)

In [79]:
df["grid_position"] = df.groupby("session_key").cumcount() + 1

In [80]:
order = [
    "meeting_key",
    "session_key",
    "session_name",
    "date",
    "time",
    "driver_number",
    "full_name",
    "team_name",
    "country",
    "lap_duration_fmt",
    "grid_position",
    "ingested_at"
]

df = df[order]
df = df.rename(columns={"lap_duration_fmt": "lap_duration"})

In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 86 entries, 8908 to 16831
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   meeting_key    86 non-null     int64              
 1   session_key    86 non-null     int64              
 2   session_name   86 non-null     object             
 3   date           86 non-null     object             
 4   time           86 non-null     object             
 5   driver_number  86 non-null     int64              
 6   full_name      86 non-null     object             
 7   team_name      86 non-null     object             
 8   country        86 non-null     object             
 9   lap_duration   86 non-null     object             
 10  grid_position  86 non-null     int64              
 11  ingested_at    86 non-null     datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), int64(4), object(7)
memory usage: 8.7+ KB


In [82]:
df

,meeting_key,session_key,session_name,date,time,driver_number,full_name,team_name,country,lap_duration,grid_position,ingested_at
8908,1279,11230,Qualifying,2026-03-07,06:17:43,63,George RUSSELL,Mercedes,United Kingdom,01:18.518,1,2026-05-04 18:23:14.237848+00:00
8905,1279,11230,Qualifying,2026-03-07,06:17:11,12,Kimi ANTONELLI,Mercedes,Italy,01:18.811,2,2026-05-04 18:23:14.237848+00:00
8911,1279,11230,Qualifying,2026-03-07,06:18:29,6,Isack HADJAR,Red Bull Racing,France,01:19.303,3,2026-05-04 18:23:14.237848+00:00
8910,1279,11230,Qualifying,2026-03-07,06:18:17,16,Charles LECLERC,Ferrari,Monaco,01:19.327,4,2026-05-04 18:23:14.237848+00:00
8909,1279,11230,Qualifying,2026-03-07,06:18:08,81,Oscar PIASTRI,McLaren,Australia,01:19.380,5,2026-05-04 18:23:14.237848+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
16772,1284,11276,Qualifying,2026-05-02,20:09:28,14,Fernando ALONSO,Aston Martin,Spain,01:31.098,18,2026-05-04 18:23:14.237848+00:00
16767,1284,11276,Qualifying,2026-05-02,20:09:08,18,Lance STROLL,Aston Martin,Canada,01:31.164,19,2026-05-04 18:23:14.237848+00:00
16840,1284,11276,Qualifying,2026-05-02,20:17:38,77,Valtteri BOTTAS,Cadillac,Finland,01:31.629,20,2026-05-04 18:23:14.237848+00:00
16836,1284,11276,Qualifying,2026-05-02,20:17:20,11,Sergio PEREZ,Cadillac,Mexico,01:31.967,21,2026-05-04 18:23:14.237848+00:00
